# 03 · Avaliação dos modelos: base → v1 → v2

Mede os três modelos no **mesmo conjunto de teste** do dataset UFPR-ALPR (1.800 imagens), com as mesmas configurações:

| Modelo | O que é |
|---|---|
| **Base** | `number-plate-yolo26s.pt`, antes do fine-tuning |
| **v1** | `best_placas_v1.pt`, fine-tuning com hiperparâmetros manuais |
| **v2** | `best_placas_v2.pt`, fine-tuning com hiperparâmetros da busca genética |

## Requisitos

- Kaggle com **GPU T4** e **Internet ligada** (painel direito → *Settings*)
- Acesso ao dataset **UFPR-ALPR**, que é restrito a pesquisa e deve ser solicitado à UFPR:
  https://web.inf.ufpr.br/vri/databases/ufpr-alpr/
- Rode as células **na ordem**, de cima para baixo

## 1) Instalar dependências

In [1]:
!pip install ultralytics huggingface_hub gdown -q

## 2) Baixar o dataset UFPR-ALPR

O dataset é baixado para `/tmp`, fora da pasta de saída do Kaggle, para não ultrapassar o limite de 20 GB.
O link abaixo só funciona quando o autor libera o acesso ao arquivo.

In [2]:
import gdown
from pathlib import Path

CAMINHO_ZIP = "/tmp/ufpr.zip"

gdown.download(
    id="1rf4RdXoaGWCPjxfv5BGG-7mFV6otLWLO",
    output=CAMINHO_ZIP,
    quiet=False,
)
print("Download concluído.")

Downloading...
From (original): https://drive.google.com/uc?id=1rf4RdXoaGWCPjxfv5BGG-7mFV6otLWLO
From (redirected): https://drive.google.com/uc?id=1rf4RdXoaGWCPjxfv5BGG-7mFV6otLWLO&confirm=t&uuid=5f3f316d-99f5-429c-8200-55e8eafe5389
To: /tmp/ufpr.zip
100%|██████████| 9.74G/9.74G [03:14<00:00, 50.2MB/s]

Download concluído.


## 3) Extrair o dataset e apagar o zip

In [3]:
import os
import zipfile

PASTA_UFPR = Path("/tmp/UFPR-ALPR dataset")

print("Extraindo...")
with zipfile.ZipFile(CAMINHO_ZIP, "r") as zf:
    zf.extractall(PASTA_UFPR.parent)
os.remove(CAMINHO_ZIP)
print(f"Pronto: {PASTA_UFPR} (zip apagado)")

Extraindo...
Pronto: /tmp/UFPR-ALPR dataset (zip apagado)


## 4) Montar a estrutura no formato YOLO

O dataset UFPR guarda as coordenadas da placa na linha `corners:` de cada `.txt`.
Aqui elas são convertidas para o formato do YOLO (`classe cx cy largura altura`, com valores de 0 a 1)
e organizadas em `images/` e `labels/`. As imagens entram como atalho (*symlink*) para não duplicar espaço.

In [4]:
PASTA_YOLO = Path("/tmp/dataset")

LARGURA, ALTURA = 1920, 1080   # resolução fixa das imagens do UFPR
splits = {"training": "train", "validation": "val", "testing": "test"}

for split_ufpr, split_yolo in splits.items():
    pasta_imgs = PASTA_YOLO / "images" / split_yolo
    pasta_lbls = PASTA_YOLO / "labels" / split_yolo
    pasta_imgs.mkdir(parents=True, exist_ok=True)
    pasta_lbls.mkdir(parents=True, exist_ok=True)

    contagem = 0
    for track in sorted((PASTA_UFPR / split_ufpr).iterdir()):
        if not track.is_dir():
            continue
        for png in sorted(track.glob("*.png")):
            txt = png.with_suffix(".txt")
            if not txt.exists():
                continue

            destino_img = pasta_imgs / png.name
            if not destino_img.exists():
                os.symlink(png, destino_img)

            label = ""
            with open(txt, encoding="utf-8") as f:
                for linha in f:
                    if linha.startswith("corners:"):
                        pares = linha.replace("corners:", "").strip().split()
                        xs = [int(p.split(",")[0]) for p in pares]
                        ys = [int(p.split(",")[1]) for p in pares]
                        cx = (min(xs) + max(xs)) / 2 / LARGURA
                        cy = (min(ys) + max(ys)) / 2 / ALTURA
                        w  = (max(xs) - min(xs)) / LARGURA
                        h  = (max(ys) - min(ys)) / ALTURA
                        label = f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n"
                        break

            (pasta_lbls / (png.stem + ".txt")).write_text(label)
            contagem += 1

    print(f"  [{split_ufpr:>10}] -> {contagem} imagens")

  [  training] -> 1800 imagens
  [validation] -> 900 imagens
  [   testing] -> 1800 imagens


## 5) Gerar o `dataset.yaml`

Diz ao YOLO onde estão as imagens e qual é a classe (uma só: `license_plate`).

In [5]:
CAMINHO_YAML = Path("/tmp/dataset.yaml")

CAMINHO_YAML.write_text(f"""path: {PASTA_YOLO}
train: images/train
val: images/val
test: images/test
nc: 1
names:
  0: license_plate
""")
print(CAMINHO_YAML.read_text())

path: /tmp/dataset
train: images/train
val: images/val
test: images/test
nc: 1
names:
  0: license_plate



## 6) Conferir o conjunto de teste

In [6]:
n_imgs   = len(list((PASTA_YOLO / "images" / "test").glob("*.png")))
n_labels = len(list((PASTA_YOLO / "labels" / "test").glob("*.txt")))
print(f"Imagens de teste : {n_imgs}")
print(f"Anotações teste  : {n_labels}")
assert n_imgs > 0, "Nenhuma imagem de teste encontrada. Revise as células 2 a 4."

Imagens de teste : 1800
Anotações teste  : 1800


## 7) Modelo base

`number-plate-yolo26s.pt`, treinado com placas de vários países, **antes** do fine-tuning. É o ponto de partida.

In [7]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

PASTA_AVALIACAO = "/kaggle/working/avaliacao"

CAMINHO_BASE = "/tmp/number-plate-yolo26s.pt"
gdown.download(
    id="1s9M8HgJQ0o4-BCnWQ3imFn_Rhsx8r-Di",
    output=CAMINHO_BASE,
    quiet=False,
)

metricas_base = YOLO(CAMINHO_BASE).val(
    data=str(CAMINHO_YAML),
    split="test",
    imgsz=1280,
    project=PASTA_AVALIACAO,
    name="base",
    exist_ok=True,
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


Downloading...
From: https://drive.google.com/uc?id=1s9M8HgJQ0o4-BCnWQ3imFn_Rhsx8r-Di
To: /tmp/number-plate-yolo26s.pt
100%|██████████| 20.3M/20.3M [00:00<00:00, 22.8MB/s]


Ultralytics 8.4.208 🚀 Python-3.13.18 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2082.0±1258.4 MB/s, size: 2143.3 KB)
val: Scanning /tmp/dataset/labels/test... 1800 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1800/1800 63.4it/s 28.4s
val: New cache created: /tmp/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 113/113 12.0s/it 22:36
                   all       1800       1800      0.863      0.898      0.896      0.602
Speed: 9.4ms preprocess, 676.6ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /kaggle/working/avaliacao/base


## 8) Modelo v1

`best_placas_v1.pt`: fine-tuning com hiperparâmetros escolhidos manualmente.

In [8]:
caminho_v1 = hf_hub_download(
    repo_id="RodrigoRRC/license-plate-br-yolo26s",
    filename="best_placas_v1.pt",
)

metricas_v1 = YOLO(caminho_v1).val(
    data=str(CAMINHO_YAML),
    split="test",
    imgsz=1280,
    project=PASTA_AVALIACAO,
    name="v1",
    exist_ok=True,
)

best_placas_v1.pt: reconstructing file:   0%|          |  0.00B / 20.4MB            

best_placas_v1.pt: downloading bytes:           |  0.00B            

Ultralytics 8.4.208 🚀 Python-3.13.18 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1554.7±325.5 MB/s, size: 2578.4 KB)
val: Scanning /tmp/dataset/labels/test.cache... 1800 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1800/1800 444.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 113/113 12.5s/it 23:33
                   all       1800       1800       0.97      0.962      0.991      0.771
Speed: 9.6ms preprocess, 712.9ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /kaggle/working/avaliacao/v1


## 9) Modelo v2

`best_placas_v2.pt`: fine-tuning com os hiperparâmetros encontrados pela busca genética.

In [9]:
caminho_v2 = hf_hub_download(
    repo_id="RodrigoRRC/license-plate-br-yolo26s",
    filename="best_placas_v2.pt",
)

metricas_v2 = YOLO(caminho_v2).val(
    data=str(CAMINHO_YAML),
    split="test",
    imgsz=1280,
    project=PASTA_AVALIACAO,
    name="v2",
    exist_ok=True,
)

best_placas_v2.pt: reconstructing file:   0%|          |  0.00B / 20.4MB            

best_placas_v2.pt: downloading bytes:           |  0.00B            

Ultralytics 8.4.208 🚀 Python-3.13.18 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 947.9±510.5 MB/s, size: 2364.1 KB)
val: Scanning /tmp/dataset/labels/test.cache... 1800 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1800/1800 139.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 113/113 11.9s/it 22:28
                   all       1800       1800      0.977      0.988      0.983       0.82
Speed: 9.5ms preprocess, 678.7ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /kaggle/working/avaliacao/v2


## 10) Evolução: base → v1 → v2

Tabela com as métricas dos três modelos. Os números também são salvos em `metricas_modelos.csv`
(aba *Output*), usado para gerar os gráficos do README.

In [10]:
import csv

modelos = {"Base": metricas_base, "v1": metricas_v1, "v2": metricas_v2}
linhas  = [("mAP@50", "map50"), ("mAP@50-95", "map"), ("Precisão", "mp"), ("Recall", "mr")]

print("=" * 58)
print(f"  {'Métrica':<12}{'Base':>14}{'v1':>14}{'v2':>14}")
print("=" * 58)
for nome, attr in linhas:
    valores = [getattr(m.box, attr) * 100 for m in modelos.values()]
    print(f"  {nome:<12}" + "".join(f"{v:>13.2f}%" for v in valores))
print("=" * 58)

with open("/kaggle/working/metricas_modelos.csv", "w", newline="") as f:
    escritor = csv.writer(f)
    escritor.writerow(["modelo", "map50", "map50_95", "precisao", "recall"])
    for nome, m in modelos.items():
        escritor.writerow([nome, m.box.map50, m.box.map, m.box.mp, m.box.mr])
print("Salvo: /kaggle/working/metricas_modelos.csv")

  Métrica               Base            v1            v2
  mAP@50              89.58%        99.11%        98.33%
  mAP@50-95           60.24%        77.09%        82.03%
  Precisão            86.32%        96.99%        97.69%
  Recall              89.77%        96.17%        98.78%
Salvo: /kaggle/working/metricas_modelos.csv
